In [2]:
import pandas as pd

In [4]:
df_listings_data = pd.read_csv("../data/London/extracted/listingsdata.csv")
df_calendar = pd.read_csv("../data/London/extracted/calendar.csv")
df_reviews_data = pd.read_csv("../data/London/extracted/reviewsdata.csv")
df_listings_info = pd.read_csv("../data/London/extracted/listings.csv")
df_reviews_info = pd.read_csv("../data/London/extracted/reviews.csv")
df_neighbourhoods = pd.read_csv("../data/London/extracted/neighbourhoods.csv")

In [5]:
datasets = {
    "listings_data": df_listings_data,
    "calendar": df_calendar,
    "reviews_data": df_reviews_data,
    "listings_info": df_listings_info,
    "reviews_info": df_reviews_info,
    "neighbourhoods": df_neighbourhoods
}

In [6]:
inventory = []

for name, df in datasets.items():

    inventory.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns)
    })

inventory_df = pd.DataFrame(inventory)

inventory_df

,dataset,rows,columns
0,listings_data,96871,79
1,calendar,35357974,7
2,reviews_data,2097996,6
3,listings_info,96871,18
4,reviews_info,2097996,2
5,neighbourhoods,33,2


In [7]:
cardinality = {}

for name, df in datasets.items():

    cardinality[name] = df.nunique()

In [9]:
cardinality_df = pd.DataFrame(cardinality)

cardinality_df

,listings_data,calendar,reviews_data,listings_info,reviews_info,neighbourhoods
accommodates,16.0,NaN,NaN,NaN,NaN,NaN
adjusted_price,NaN,0.0,NaN,NaN,NaN,NaN
amenities,85530.0,NaN,NaN,NaN,NaN,NaN
availability_30,31.0,NaN,NaN,NaN,NaN,NaN
availability_365,366.0,NaN,NaN,366.0,NaN,NaN
...,...,...,...,...,...,...
reviewer_name,NaN,NaN,205732.0,NaN,NaN,NaN
reviews_per_month,910.0,NaN,NaN,910.0,NaN,NaN
room_type,4.0,NaN,NaN,4.0,NaN,NaN
scrape_id,1.0,NaN,NaN,NaN,NaN,NaN


In [10]:
null_summary = []

for dataset_name, df in datasets.items():

    for column in df.columns:

        null_pct = (
            df[column]
            .isnull()
            .mean()
            * 100
        )

        null_summary.append({
            "dataset": dataset_name,
            "column": column,
            "null_pct": round(null_pct,2)
        })

null_df = pd.DataFrame(null_summary)

In [11]:
null_df.sort_values(
    "null_pct",
    ascending=False
)

,dataset,column,null_pct
49,listings_data,calendar_updated,100.0
29,listings_data,neighbourhood_group_cleansed,100.0
109,listings_info,license,100.0
83,calendar,adjusted_price,100.0
82,calendar,price,100.0
...,...,...,...
106,listings_info,calculated_host_listings_count,0.0
108,listings_info,number_of_reviews_ltm,0.0
110,reviews_info,listing_id,0.0
111,reviews_info,date,0.0


In [12]:
for name, df in datasets.items():

    print(name)

    print(
        df.dtypes.value_counts()
    )

listings_data
str        34
float64    25
int64      20
Name: count, dtype: int64
calendar
int64      3
str        2
float64    2
Name: count, dtype: int64
reviews_data
int64    3
str      3
Name: count, dtype: int64
listings_info
int64      7
float64    6
str        5
Name: count, dtype: int64
reviews_info
int64    1
str      1
Name: count, dtype: int64
neighbourhoods
float64    1
str        1
Name: count, dtype: int64


In [14]:
dtype_summary = []

for dataset_name, df in datasets.items():

    counts = (
        df.dtypes
        .astype(str)
        .value_counts()
    )

    for dtype, count in counts.items():

        dtype_summary.append({
            "dataset": dataset_name,
            "dtype": dtype,
            "count": count
        })

dtype_df = pd.DataFrame(dtype_summary)

dtype_df

,dataset,dtype,count
0,listings_data,str,34
1,listings_data,float64,25
2,listings_data,int64,20
3,calendar,int64,3
4,calendar,str,2
5,calendar,float64,2
6,reviews_data,int64,3
7,reviews_data,str,3
8,listings_info,int64,7
9,listings_info,float64,6


In [18]:
for name, df in datasets.items():

    numeric_df = df.select_dtypes(
        include="number"
    )

    stats = numeric_df.describe()

    stats.to_csv(
        f"../data/London/profiling/{name}_stats.csv"
    )

## GeoJson Exploration

In [28]:
import json

with open("../data/London/extracted/neighbourhoods.geojson", "r", encoding="utf-8") as f:
    geo = json.load(f)

print("GeoJSON Type:", geo["type"])
print("Number of Features:", len(geo["features"]))

GeoJSON Type: FeatureCollection
Number of Features: 33


In [29]:
first_feature = geo["features"][0]

print(first_feature.keys())

dict_keys(['type', 'geometry', 'properties'])


In [30]:
geometry_types = set()

for feature in geo["features"]:
    geometry_types.add(
        feature["geometry"]["type"]
    )

print(geometry_types)

{'MultiPolygon'}


In [31]:
geo["features"][0]["properties"]

{'neighbourhood': 'Kingston upon Thames', 'neighbourhood_group': None}

In [32]:
property_fields = geo["features"][0]["properties"].keys()

print(list(property_fields))

['neighbourhood', 'neighbourhood_group']


In [33]:
geo_profile = {
    "geojson_type": geo["type"],
    "feature_count": len(geo["features"]),
    "geometry_types": list(geometry_types),
    "property_fields": list(property_fields)
}

geo_profile

{'geojson_type': 'FeatureCollection',
 'feature_count': 33,
 'geometry_types': ['MultiPolygon'],
 'property_fields': ['neighbourhood', 'neighbourhood_group']}

In [36]:
with pd.ExcelWriter(
    "../data/London/profiling/data_profiling_report.xlsx"
) as writer:

    inventory_df.to_excel(
        writer,
        sheet_name="Inventory",
        index=False
    )

    null_df.to_excel(
        writer,
        sheet_name="Null Summary",
        index=False
    )

    dtype_df.to_excel(
        writer,
        sheet_name="Data Types",
        index=False
    )

    cardinality_df.to_excel(
        writer,
        sheet_name="Cardinality"
    )

## Duplicate Detection

In [43]:
duplicate_summary = []

# listings_data
duplicate_summary.append({
    "dataset": "listings_data",
    "check_type": "Primary Key (id)",
    "duplicate_count": df_listings_data["id"].duplicated().sum()
})

duplicate_summary.append({
    "dataset": "listings_data",
    "check_type": "Full Row",
    "duplicate_count": df_listings_data.duplicated().sum()
})

# listings_info
duplicate_summary.append({
    "dataset": "listings_info",
    "check_type": "Primary Key (id)",
    "duplicate_count": df_listings_info["id"].duplicated().sum()
})

duplicate_summary.append({
    "dataset": "listings_info",
    "check_type": "Full Row",
    "duplicate_count": df_listings_info.duplicated().sum()
})

# reviews_data
duplicate_summary.append({
    "dataset": "reviews_data",
    "check_type": "Primary Key (id)",
    "duplicate_count": df_reviews_data["id"].duplicated().sum()
})

duplicate_summary.append({
    "dataset": "reviews_data",
    "check_type": "Full Row",
    "duplicate_count": df_reviews_data.duplicated().sum()
})

# reviews_info
duplicate_summary.append({
    "dataset": "reviews_info",
    "check_type": "Composite Key (listing_id, date)",
    "duplicate_count": df_reviews_info.duplicated(
        subset=["listing_id", "date"]
    ).sum()
})

duplicate_summary.append({
    "dataset": "reviews_info",
    "check_type": "Full Row",
    "duplicate_count": df_reviews_info.duplicated().sum()
})

# calendar
duplicate_summary.append({
    "dataset": "calendar",
    "check_type": "Composite Key (listing_id, date)",
    "duplicate_count": df_calendar.duplicated(
        subset=["listing_id", "date"]
    ).sum()
})

duplicate_summary.append({
    "dataset": "calendar",
    "check_type": "Full Row",
    "duplicate_count": df_calendar.duplicated().sum()
})

# neighbourhoods
duplicate_summary.append({
    "dataset": "neighbourhoods",
    "check_type": "Neighbourhood Name",
    "duplicate_count": df_neighbourhoods["neighbourhood"].duplicated().sum()
})

duplicate_summary.append({
    "dataset": "neighbourhoods",
    "check_type": "Full Row",
    "duplicate_count": df_neighbourhoods.duplicated().sum()
})

In [44]:
duplicate_summary_df = pd.DataFrame(duplicate_summary)

duplicate_summary_df

,dataset,check_type,duplicate_count
0,listings_data,Primary Key (id),0
1,listings_data,Full Row,0
2,listings_info,Primary Key (id),0
3,listings_info,Full Row,0
4,reviews_data,Primary Key (id),0
5,reviews_data,Full Row,0
6,reviews_info,"Composite Key (listing_id, date)",12743
7,reviews_info,Full Row,12743
8,calendar,"Composite Key (listing_id, date)",0
9,calendar,Full Row,0


## Fuzzy matching

In [46]:
from rapidfuzz import fuzz
from itertools import combinations

values = (
    df_listings_data["property_type"]
    .dropna()
    .unique()
)

potential_duplicates = []

for a, b in combinations(values, 2):

    score = fuzz.ratio(a, b)

    if score >= 90:
        potential_duplicates.append(
            [a, b, score]
        )

fuzzy_df = pd.DataFrame(
    potential_duplicates,
    columns=["value_a", "value_b", "similarity"]
)

fuzzy_df

,value_a,value_b,similarity
0,Private room in home,Private room in hostel,90.476190
1,Private room in loft,Private room in boat,90.000000
2,Private room in yurt,Private room in hut,92.307692
3,Room in hotel,Room in hostel,96.296296
4,Private room in chalet,Private room in castle,90.909091
5,Private room in boat,Private room in barn,90.000000
6,Shared room in hostel,Shared room in home,90.000000
7,Shared room in hostel,Shared room in hotel,97.560976
8,Shared room in home,Shared room in hotel,92.307692


In [49]:
values = (
    df_listings_data["host_location"]
    .dropna()
    .unique()
)

potential_duplicates = []

for a, b in combinations(values, 2):

    score = fuzz.ratio(a, b)

    if score >= 90:
        potential_duplicates.append(
            [a, b, score]
        )

fuzzy_df = pd.DataFrame(
    potential_duplicates,
    columns=["value_a", "value_b", "similarity"]
)

fuzzy_df

,value_a,value_b,similarity
0,"Morden, United Kingdom","Rode, United Kingdom",90.476190
1,"Feltham, United Kingdom","Fareham, United Kingdom",91.304348
2,"Twickenham, United Kingdom","Beckenham, United Kingdom",90.196078
3,"Twickenham, United Kingdom","Ickenham, United Kingdom",92.000000
4,"Kensington, United Kingdom","Ossington, United Kingdom",90.196078
...,...,...,...
470,"Barnham, United Kingdom","Earsham, United Kingdom",91.304348
471,"East Preston, United Kingdom","East Meon, United Kingdom",90.566038
472,"Fareham, United Kingdom","Earsham, United Kingdom",91.304348
473,"Writtle, United Kingdom","Witley, United Kingdom",93.333333


In [ ]:
missing_summary = []  # Detecting missing values

for dataset_name, df in datasets.items():

    missing_pct = (
        df.isnull()
        .mean()
        .mul(100)
        .round(2)
        .sort_values(ascending=False)
    )

    for column, pct in missing_pct.items():

        missing_summary.append({
            "dataset": dataset_name,
            "column": column,
            "missing_pct": pct
        })

missing_df = pd.DataFrame(missing_summary)

In [ ]:
missing_df.sort_values(  # top columns with highest missing values
    "missing_pct",
    ascending=False
).head(20)

,dataset,column,missing_pct
0,listings_data,license,100.00
1,listings_data,calendar_updated,100.00
2,listings_data,neighbourhood_group_cleansed,100.00
93,listings_info,neighbourhood_group,100.00
79,calendar,price,100.00
112,neighbourhoods,neighbourhood_group,100.00
92,listings_info,license,100.00
80,calendar,adjusted_price,100.00
4,listings_data,neighborhood_overview,57.46
3,listings_data,neighbourhood,57.46


In [53]:
def detect_outliers_iqr(df, column):  # outlier detection

    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = df[
        (df[column] < lower)
        | (df[column] > upper)
    ]

    return outliers

In [55]:
df_listings_data["price"].head()

0     $70.00
1    $149.00
2    $411.00
3        NaN
4    $210.00
Name: price, dtype: str

In [56]:
df_listings_data["price_numeric"] = (
    df_listings_data["price"]
    .replace(r"[\$,]", "", regex=True)
    .astype(float)
)

In [57]:
df_listings_data["price_numeric"].dtype

dtype('float64')

In [58]:
price_outliers = detect_outliers_iqr(
    df_listings_data,
    "price_numeric"
)

len(price_outliers)

4195

In [62]:
len(price_outliers) / len(df_listings_data) * 100

4.330501388444427

In [59]:
availability_365_outliers = detect_outliers_iqr(
    df_listings_data,
    "availability_365"
)

len(availability_365_outliers)

0

In [60]:
number_of_reviews_outliers = detect_outliers_iqr(
    df_listings_data,
    "number_of_reviews"
)

len(number_of_reviews_outliers)

11446

In [63]:
len(number_of_reviews_outliers) / len(df_listings_data) * 100

11.815713681081025

In [61]:
minimum_nights_outliers = detect_outliers_iqr(
    df_listings_data,
    "minimum_nights"
)

len(minimum_nights_outliers)

7136

In [64]:
len(minimum_nights_outliers) / len(df_listings_data) * 100

7.366497713453975

## validation checks

In [65]:
validation_results = []

In [66]:
negative_price_count = len(
    df_listings_data[
        df_listings_data["price_numeric"] < 0
    ]
)

validation_results.append({
    "rule": "Price cannot be negative",
    "violations": negative_price_count
})


invalid_latitude_count = len(
    df_listings_data[
        (df_listings_data["latitude"] < -90) |
        (df_listings_data["latitude"] > 90)
    ]
)

validation_results.append({
    "rule": "Latitude must be between -90 and 90",
    "violations": invalid_latitude_count
})

invalid_longitude_count = len(
    df_listings_data[
        (df_listings_data["longitude"] < -180) |
        (df_listings_data["longitude"] > 180)
    ]
)

validation_results.append({
    "rule": "Longitude must be between -180 and 180",
    "violations": invalid_longitude_count
})

invalid_availability_count = len(
    df_listings_data[
        (df_listings_data["availability_365"] < 0) |
        (df_listings_data["availability_365"] > 365)
    ]
)

validation_results.append({
    "rule": "Availability_365 must be between 0 and 365",
    "violations": invalid_availability_count
})


invalid_minimum_nights_count = len(
    df_listings_data[
        df_listings_data["minimum_nights"] <= 0
    ]
)

validation_results.append({
    "rule": "Minimum nights must be greater than 0",
    "violations": invalid_minimum_nights_count
})

invalid_reviews_count = len(
    df_listings_data[
        df_listings_data["number_of_reviews"] < 0
    ]
)

validation_results.append({
    "rule": "Number of reviews cannot be negative",
    "violations": invalid_reviews_count
})

invalid_host_count = len(
    df_listings_data[
        df_listings_data["host_listings_count"] < 0
    ]
)

validation_results.append({
    "rule": "Host listings count cannot be negative",
    "violations": invalid_host_count
})

In [67]:
validation_df = pd.DataFrame(validation_results)

validation_df


,rule,violations
0,Price cannot be negative,0
1,Latitude must be between -90 and 90,0
2,Longitude must be between -180 and 180,0
3,Availability_365 must be between 0 and 365,0
4,Minimum nights must be greater than 0,0
5,Number of reviews cannot be negative,0
6,Host listings count cannot be negative,0
